# SEMTM0009 Week 3 Worksheet — computational template

**Parameterising and analysing heterogeneous models.**

The plumbing is written. What is left to you is marked `# TODO`, and each
`# TODO` returns `np.nan` so the whole notebook runs at any stage and shows you
what is still missing.

**Units.** The survey counted contacts **per week**, so every rate from it is
per week. Question 3 works in **days**. Never mix them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "lines.linewidth": 1.8,
                     "axes.spines.top": False, "axes.spines.right": False})
np.set_printoptions(precision=4, suppress=True)

# ------------------------------------------------------------------ DATA -----
# Reads data/ next to this notebook if it is there (Jupyter, VS Code);
# otherwise, e.g. in Google Colab, fetches the same files from the course repo.
import os
DATA = ("data" if os.path.isdir("data") else
        "https://raw.githubusercontent.com/SEMTM0009-University-of-Bristol/notebooks/main/Week3/data")

survey = pd.read_csv(f"{DATA}/contact_survey.csv")   # 250 rows: status, count, who each contact is
counts = pd.read_csv(f"{DATA}/contact_counts.csv")   # 250 rows: status and count only
sero   = pd.read_csv(f"{DATA}/serology_age.csv")     # 1000 rows: age, seropositive

print("survey :", survey.shape, "| ID participants:", (survey.UserStatus == "ID").sum())
print("counts :", counts.shape, "| mean contacts:", round(counts.NumberOfContacts.mean(), 4))
print("sero   :", sero.shape, "| children:", (sero.age_years < 18).sum())
display(survey.head(3))

---
# Question 1 — the town with the full survey

Groups are indexed `0 = ID` (high risk, injects drugs) and `1 = NID` (low risk).
Every matrix in this notebook uses that order, and rows are **"to"**, columns
are **"from"**.

In [ ]:
GROUPS = ["ID", "NID"]

def group_totals(survey):
    '''Total contacts reported BY group i WITH group j, from the survey.

    Each row lists NumberOfContacts and then that many contact columns; the
    remaining columns are NaN.  Only the first NumberOfContacts entries count.
    A contact entry is 1 if that contact injects drugs, 0 if not.
    '''
    cc = [c for c in survey.columns if c.startswith("Contact")]
    tot = np.zeros((2, 2))
    for _, row in survey.iterrows():
        k = int(row.NumberOfContacts)
        n_ID = np.nansum(row[cc].to_numpy(dtype=float)[:k])
        i = GROUPS.index(row.UserStatus)
        tot[i, 0] += n_ID          # contacts with ID
        tot[i, 1] += k - n_ID      # contacts with NID
    return tot


def group_sizes(df):
    n = np.array([(df.UserStatus == g).sum() for g in GROUPS], dtype=float)
    return n / n.sum()


C = group_totals(survey)
n = group_sizes(survey)
N = len(survey)
print("total contacts C =\n", C)
print("group sizes n =", n)

In [ ]:
def contact_rate_matrix(C, n, N):
    '''Q1(c): turn contact COUNTS into a per-pair RATE.

    Divide each element by the expected number of pairs between the groups,
    N n_i n_j, with n_i a FRACTION.  Units: per week.
    '''
    # TODO: replace np.nan with your expression.
    return np.nan * C

r_raw = contact_rate_matrix(C, n, N)
print("r (raw) =\n", r_raw)
print("\noff-diagonal:", r_raw[0, 1], "vs", r_raw[1, 0])

### Q1(d)–(e) — reciprocity

**Question.** What must be true of $r_{HL}$ and $r_{LH}$, and why is it not
true here? Draw a small network by hand and count from both ends if you are not
sure.

*Your answer:*

In [ ]:
def symmetrise(r):
    '''Q1(e): the reciprocity repair.'''
    # TODO
    return np.nan * r

r_hat = symmetrise(r_raw)
print("r_hat =\n", r_hat)

---
# Question 2 — the town with contact counts only

In [ ]:
c = np.array([counts.NumberOfContacts[counts.UserStatus == g].mean() for g in GROUPS])
q = group_sizes(counts)
c_bar = c @ q                       # <c>, the population mean contact number
print(f"group mean contacts c = {c},  group sizes q = {q}")
print(f"<c> = {c_bar:.4f}")


def proportional_matrix(c, c_bar):
    '''Q2(a): random mixing at a rate proportional to contact numbers.'''
    # TODO
    return np.nan * np.outer(c, c)

r_prop = proportional_matrix(c, c_bar)
print("\nr (proportional) =\n", r_prop)

**Question (a).** Two separate assumptions went into that matrix. Name both.

*Your answer:*

In [ ]:
def assortativity(beta, n):
    '''Q2(b): measure Q from a matrix you already have.

    beta_hat_ij = beta_ij * n_j is how many of group i's contacts are with
    group j; B_ii is the share of group i's contacts that stay within the group.
    '''
    beta_hat = beta * n[None, :]
    B = beta_hat / beta_hat.sum(axis=1, keepdims=True)
    m = beta.shape[0]
    # TODO: combine the diagonal of B into Q
    return np.nan

Q = assortativity(r_hat, n)
print(f"Q measured from Question 1's corrected matrix = {Q}")

In [ ]:
def assortative_matrix(c, q, c_bar, Q):
    '''Q2(b): blend within-group mixing into the proportional matrix.'''
    r = np.empty((2, 2))
    for i in range(2):
        for j in range(2):
            # TODO: diagonal and off-diagonal are different -- see the worksheet
            r[i, j] = np.nan
    return r

r_asso = assortative_matrix(c, q, c_bar, Q)
print("r (proportional + measured Q) =\n", r_asso)

### Q2(c) — $R_0$ for both matrices

$1/\gamma = 14$ days $= 2$ **weeks**, because the contact data are per week.

In [ ]:
INV_GAMMA_WEEKS = 2.0

def next_generation_matrix(beta, n, inv_gamma=INV_GAMMA_WEEKS):
    '''K_ij = (1/gamma) * n_i * beta_ij.

    The n_i weights the ROW: it came from S_i = n_i at the disease-free
    equilibrium, so it belongs to the group being INFECTED.
    '''
    # TODO
    return np.nan * beta


def R0_from_K(K):
    '''Dominant eigenvalue: the largest in absolute value.'''
    # TODO
    return np.nan

for label, mat in [("proportional", r_prop), ("assortative", r_asso)]:
    K = next_generation_matrix(mat, q)
    print(f"{label:14s}: K =\n{K}\n{'':14s}  R0 = rho(K) = {R0_from_K(K)}")

---
# Question 3 — an age-structured model from serology

Now consider an age-based SIR model (with demographics) of an endemic disease
with the transmission matrix

$$\boldsymbol\beta=\begin{pmatrix}\beta_{CC} & \beta_{CA}\\ \beta_{AC} & \beta_{AA}\end{pmatrix},
\qquad \beta_{CA}=\beta_{AC}=\beta_{AA}.$$

Using the serological data in `data/serology_age.csv`, estimate the values of
$\boldsymbol\beta$.

- Children are under 18; life expectancy 70 years, so $T_C=18$ and $T_A=52$
  years, with $\mu_A=1/T_A$ and $\mu_C=0$.
- Infectious period 14 days. Work in **days**.
- $S_C^*$ and $S_A^*$ are fractions of the **whole sample**, not of their own
  age group.

There is a worked example of this in the lecture slides.


In [ ]:
GAMMA_PER_DAY = 1/14
CHILD_AGE_LIMIT = 18
TC = 18*365                  # days in the child class
TA = 52*365                  # days in the adult class (life expectancy 70 y)
MU_A_PER_DAY = 1/TA          # adult death rate; children do not die (mu_C = 0)
lC = 1/TC                    # ageing rate

def serology_summary(df):
    '''n_C, n_A, S_C*, S_A* -- all as fractions of the WHOLE sample.'''
    N = len(df)
    child = df.age_years < CHILD_AGE_LIMIT
    nC, nA = child.sum()/N, (~child).sum()/N
    SC = ((df.seropositive == 0) & child).sum()/N
    SA = ((df.seropositive == 0) & ~child).sum()/N
    return nC, nA, SC, SA

nC, nA, SC, SA = serology_summary(sero)
print(f"nC={nC:.4f}  nA={nA:.4f}  SC={SC:.4f}  SA={SA:.4f}")


### 💻 The equilibrium infected fractions

> **How to start.** At equilibrium the fraction infected in a class is the flow in
> times the time spent infected:
> $$I_C^*=\frac{1}{\gamma T_C}\left(n_C-S_C^*\right),\qquad
>   I_A^*=\frac{1}{\gamma T_A}\cdot\frac{S_C^*n_A-S_A^*n_C}{n_C}.$$


In [ ]:
def infected_equilibria(nC, nA, SC, SA, gamma=GAMMA_PER_DAY, TC=TC, TA=TA):
    '''Q3: reconstruct I_C* and I_A* from the serology.'''
    # TODO: two expressions, from the hint above
    IC = np.nan
    IA = np.nan
    return IC, IA

IC, IA = infected_equilibria(nC, nA, SC, SA)
print(f"IC* = {IC:.6e}   IA* = {IA:.6e}")


### 💻 Solve for $\beta_{CC}$ and $\beta_{CA}=\beta_{AC}=\beta_{AA}$

> **How to start.** With $\mu_C=0$, $\dot I_C=0$ and $\dot I_A=0$ give
> $$\frac{(\gamma+\ell_C)I_C^*}{S_C^*}=\beta_{CC} I_C^*+\beta_{CA} I_A^*,\qquad
>   \frac{(\gamma+\mu_A)I_A^*-\ell_C I_C^*}{S_A^*}=\beta_{AC} I_C^*+\beta_{AA} I_A^*.$$
> With $\beta_{AC}=\beta_{AA}=\beta_{CA}$ the second gives $\beta_{CA}$ directly;
> put it into the first for $\beta_{CC}$.


In [ ]:
def solve_betas(IC, IA, SC, SA, gamma=GAMMA_PER_DAY, mu=MU_A_PER_DAY, lC=lC):
    '''Q3: two equilibrium conditions, two unknowns (mu = mu_A, mu_C = 0).

    (1)  (gamma+lC) IC / SC           = b_CC IC + b_CA IA
    (2)  ((gamma+mu) IA - lC IC) / SA = b_AC IC + b_AA IA = b_CA (IC + IA)
    (2) gives b_CA directly; put it into (1) for b_CC.
    '''
    # TODO
    b_CC = np.nan
    b_CA = np.nan
    return b_CC, b_CA

b_CC, b_CA = solve_betas(IC, IA, SC, SA)
print(f"beta_CC = {b_CC}   beta_CA = beta_AC = beta_AA = {b_CA}   (per day)")
